# Week 61, Day 4: t-SNE: neighbour embeddings

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w061/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
g0 = np.random.default_rng(614)
centres = 2.5 * g0.normal(size=(5, 50))
centres[4] *= 3                                        # class 4 is far from the others
spread = np.array([1.0, 1.0, 3.0, 1.0, 1.0])           # class 2 is three times as spread
labels = np.repeat(np.arange(5), 60)
Xt = centres[labels] + spread[labels, None] * g0.normal(size=(300, 50))

def sqdist(A):
    s = (A ** 2).sum(1)
    return np.maximum(s[:, None] + s[None] - 2 * A @ A.T, 0)

def knn_class_acc(Y, lab, k=10):
    D = sqdist(Y); np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, axis=1)[:, :k]
    return np.mean((lab[nn] == lab[:, None]).mean(1))
print("ready: 300 trials, 50 neurons, 5 stimulus classes")

### Exercise: Perplexity-calibrated affinities

Write cond_probs(D2, perp), which for each row finds by bisection the precision beta = 1/(2 sigma squared) that gives entropy log(perp) (in nats), and returns the conditional probabilities. Then symmetrise into joint probabilities.

In [ ]:
def cond_probs(D2, perp, tol=1e-5):
    n = len(D2); P = np.zeros((n, n)); target = np.log(perp)
    for i in range(n):
        d = np.delete(D2[i], i)
        lo, hi = 0.0, np.inf; beta = 1.0 / np.median(d)
        for _ in range(100):
            p = np.exp(-(d - d.min()) * beta); p /= p.sum()
            H = -(p * np.log(np.maximum(p, 1e-300))).sum()
            if abs(H - target) < tol:
                break
            if H > target:                                  # too many neighbours: narrow the kernel
                lo = beta; beta = beta * 2 if hi == np.inf else (beta + hi) / 2
            else:
                hi = beta; beta = (beta + lo) / 2
        P[i, np.arange(n) != i] = p
    return P

D2 = sqdist(Xt)
Pc = cond_probs(D2, 30)
Pj = ...
H_rows = -(Pc * np.log(np.maximum(Pc, 1e-300))).sum(1)
print(f"row entropies from {H_rows.min():.4f} to {H_rows.max():.4f} nats (target {np.log(30):.4f})")
print(f"largest affinity per row: class 2 (spread) {Pc[labels == 2].max(1).mean():.3f}, class 0 {Pc[labels == 0].max(1).mean():.3f}")

In [ ]:
import numpy as np
def test_perp():
    "each row of conditional probabilities has entropy log(perplexity) to within 1e-3"
    H = -(Pc * np.log(np.maximum(Pc, 1e-300))).sum(1)
    assert np.allclose(H, np.log(30), atol=1e-3)
def test_joint():
    "the joint matrix is symmetric and sums to one"
    assert np.allclose(Pj, Pj.T) and abs(Pj.sum() - 1) < 1e-9

test_perp()
test_joint()
print("All checks passed")

### Exercise: t-SNE by gradient descent

Write tsne_grad(P, Y) from the formula above, then run 500 iterations of gradient descent with momentum (early exaggeration of 12 for the first 100) and compare 10-nearest-neighbour class agreement in the embedding with that in the top two principal components.

In [ ]:
def tsne_grad(P, Y):
    W = 1 / (1 + sqdist(Y)); np.fill_diagonal(W, 0)
    Q = W / W.sum()
    M = ...                                             # (p_ij - q_ij) / (1 + d_ij^2)
    return 4 * (M.sum(1)[:, None] * Y - M @ Y)

def run_tsne(P, seed=0, iters=500, lr=100.0):
    g = np.random.default_rng(seed)
    Y = 1e-4 * g.normal(size=(len(P), 2)); V = np.zeros_like(Y)
    for it in range(iters):
        exag, mom = (12.0, 0.5) if it < 100 else (1.0, 0.8)
        V = mom * V - lr * tsne_grad(exag * P, Y)
        Y = Y + V
    return Y

Y_tsne = run_tsne(Pj)
Xc = Xt - Xt.mean(0)
Y_pca = Xc @ np.linalg.svd(Xc, full_matrices=False)[2][:2].T
acc_tsne, acc_pca = knn_class_acc(Y_tsne, labels), knn_class_acc(Y_pca, labels)
print(f"10-nearest-neighbour class agreement: t-SNE {acc_tsne:.3f}, PCA {acc_pca:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(*Y_pca.T, c=labels, cmap="tab10", s=8); ax[0].set_title("PCA")
ax[1].scatter(*Y_tsne.T, c=labels, cmap="tab10", s=8); ax[1].set_title("t-SNE, perplexity 30"); plt.show()

In [ ]:
import numpy as np
def test_grad():
    "tsne_grad matches a finite-difference gradient of the KL divergence"
    g = np.random.default_rng(0); Ys = g.normal(size=(6, 2)); Ps = g.random((6, 6)); Ps = Ps + Ps.T; np.fill_diagonal(Ps, 0); Ps /= Ps.sum()
    def kl(Yf):
        W = 1 / (1 + sqdist(Yf)); np.fill_diagonal(W, 0); Q = W / W.sum()
        m = Ps > 0
        return np.sum(Ps[m] * np.log(Ps[m] / Q[m]))
    num = np.zeros_like(Ys)
    for i in range(6):
        for k in range(2):
            E = np.zeros_like(Ys); E[i, k] = 1e-6
            num[i, k] = (kl(Ys + E) - kl(Ys - E)) / 2e-6
    assert np.allclose(tsne_grad(Ps, Ys), num, atol=1e-5)
def test_quality():
    "neighbours in the t-SNE embedding share their class more often than in the PCA projection"
    assert acc_tsne > acc_pca and acc_tsne > 0.95

test_grad()
test_quality()
print("All checks passed")

### Exercise: Sizes and distances in the embedding

Measure the spread of each class (root mean squared distance to its centroid) in the data and in the embedding, and compare class 2 with class 0. Then compare the distance from class 4's centroid to the centroid of classes 0 to 3, relative to the same distance for class 0, in the data and in the embedding.

In [ ]:
def spread_of(Z, c):
    Zc = Z[labels == c]
    return np.sqrt(((Zc - Zc.mean(0)) ** 2).sum(1).mean())

def far_ratio(Z):
    cents = np.array([Z[labels == c].mean(0) for c in range(5)])
    rest = cents[:4].mean(0)
    return np.linalg.norm(cents[4] - rest) / np.linalg.norm(cents[0] - rest)

true_ratio = spread_of(Xt, 2) / spread_of(Xt, 0)
emb_ratio = ...
true_far, emb_far = far_ratio(Xt), far_ratio(Y_tsne)
print(f"spread of class 2 relative to class 0: data {true_ratio:.2f}, t-SNE {emb_ratio:.2f}")
print(f"distance of class 4 from the rest, relative to class 0: data {true_far:.2f}, t-SNE {emb_far:.2f}")

In [ ]:
import numpy as np
def test_size():
    "class 2 is three times as spread in the data but less than twice as spread in the embedding"
    assert true_ratio > 2.5 and emb_ratio < 2.0
def test_far():
    "class 4 is several times farther from the rest than class 0 in the data, but much less so in the embedding"
    assert true_far > 2.0 and emb_far < 0.6 * true_far

test_size()
test_far()
print("All checks passed")